# Stage 1: Define the Problem Statement and Solve it Yourself

**Problem Statement**: Time management is important but cumbersome, and not having a less tedious way of doing this has caused many to not be able to make use of their limited hours.

**Proposed Solution(Data Flow from solving it myself)**: Translate traces into emails/invites, and then surface the emails/invites in a preview, and only if a human approves the preview, is the email/invite sent.

Scope: Inputs of whatsapp messages(for now)

# Stage 2: Define the Steps and Build a System

A 5 Step process(workflow)

1. A LLM non-Looping Agent that translates traces into Tool Calls of the following two tools:
* Outlook Email Writer Tool  
Inputs: 

* Outlook Calendar Invite Writer Tool

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../../.env')
os.getenv("MICROSOFT_MCP_TENANT_ID")

'consumers'

In [2]:
os.getenv("MICROSOFT_MCP_CLIENT_ID")


'api://4196953a-89de-43f7-8317-0391857f0124'

In [3]:
import os, re, asyncio
from fastmcp import Client
from fastmcp.client.transports import UvxStdioTransport  # runs a python package via `uvx`

APP_ID = os.getenv("MICROSOFT_MCP_CLIENT_ID")
TENANT = os.getenv("MICROSOFT_MCP_TENANT_ID")

async def get_tools_and_linked_accounts(app_id: str, tenant: str):
    # 1) Use stdio to spawn the server via `uvx` (no local clone needed)
    transport = UvxStdioTransport(
        from_package="git+https://github.com/elyxlz/microsoft-mcp.git",
        tool_name="microsoft-mcp",
        env_vars={
            "MICROSOFT_MCP_CLIENT_ID": app_id,
            "MICROSOFT_MCP_TENANT_ID": tenant,
        },
    )
    client = Client(transport)

    async with client:
        # Optional: see what's available
        tools = await client.list_tools()
        print("Tools:", [t.name for t in tools])
        print(tools)


        # 5) Sanity check: list accounts
        accounts = await client.call_tool("list_accounts", {})
        print("\nLinked accounts:", accounts.content[0].text)
        return tools, accounts

tools, linked_accounts = await get_tools_and_linked_accounts(APP_ID, TENANT)



Tools: ['list_accounts', 'authenticate_account', 'complete_authentication', 'list_emails', 'get_email', 'create_email_draft', 'send_email', 'update_email', 'delete_email', 'move_email', 'reply_to_email', 'reply_all_email', 'list_events', 'get_event', 'create_event', 'update_event', 'delete_event', 'respond_event', 'check_availability', 'list_contacts', 'get_contact', 'create_contact', 'update_contact', 'delete_contact', 'list_files', 'get_file', 'create_file', 'update_file', 'delete_file', 'get_attachment', 'search_files', 'search_emails', 'search_events', 'search_contacts', 'unified_search']
[Tool(name='list_accounts', title=None, description='List all signed-in Microsoft accounts', inputSchema={'properties': {}, 'type': 'object'}, outputSchema={'properties': {'result': {'items': {'additionalProperties': {'type': 'string'}, 'type': 'object'}, 'title': 'Result', 'type': 'array'}}, 'required': ['result'], 'title': '_WrappedResult', 'type': 'object', 'x-fastmcp-wrap-result': True}, annot

In [4]:
tools

[Tool(name='list_accounts', title=None, description='List all signed-in Microsoft accounts', inputSchema={'properties': {}, 'type': 'object'}, outputSchema={'properties': {'result': {'items': {'additionalProperties': {'type': 'string'}, 'type': 'object'}, 'title': 'Result', 'type': 'array'}}, 'required': ['result'], 'title': '_WrappedResult', 'type': 'object', 'x-fastmcp-wrap-result': True}, annotations=None, meta={'_fastmcp': {'tags': []}}),
 Tool(name='authenticate_account', title=None, description='Authenticate a new Microsoft account using device flow authentication\n\nReturns authentication instructions and device code for the user to complete authentication.\nThe user must visit the URL and enter the code to authenticate their Microsoft account.', inputSchema={'properties': {}, 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'string'}, 'type': 'object'}, annotations=None, meta={'_fastmcp': {'tags': []}}),
 Tool(name='complete_authentication', title=None, descrip

In [5]:
linked_accounts

CallToolResult(content=[TextContent(type='text', text='[{"username":"mnylnworkshopparticipant@outlook.com","account_id":"00000000-0000-0000-b358-02ef6301080d.9188040d-6c67-4c5b-b112-36a304b66dad"}]', annotations=None, meta=None)], structured_content={'result': [{'username': 'mnylnworkshopparticipant@outlook.com', 'account_id': '00000000-0000-0000-b358-02ef6301080d.9188040d-6c67-4c5b-b112-36a304b66dad'}]}, data=[Root()], is_error=False)

In [6]:
import ast
account_id =ast.literal_eval(linked_accounts.content[0].text)[0]['account_id']
account_id


'00000000-0000-0000-b358-02ef6301080d.9188040d-6c67-4c5b-b112-36a304b66dad'

In [7]:
# pip install google-genai mcp
import asyncio
import json
from typing import Dict, Any

from google import genai
from google.genai import types

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import ast
from datetime import datetime

# ---- 1) Define which MCP tools you want to expose ----
ALLOWED = {
    "list_events",        # <-- change to your tool names
    "get_event",
    "create_event",
}

account_id = ast.literal_eval(linked_accounts.content[0].text)[0]['account_id']
system_instruction = f"You are a helpful assistant that helps users manage their outlook calendars. You are provided with a list of tools that you can use to help the user. Note that the account ID is {account_id}, and Today is {datetime.now().strftime('%Y-%m-%d')}."


def _sanitize_openapi_schema(schema: Dict[str, Any]) -> Dict[str, Any]:
    """
    Gemini accepts a subset of OpenAPI 3.0 for function parameters.
    Strip fields that often appear in JSON Schema but aren’t needed.
    """
    if not schema:
        return {"type": "object", "properties": {}}
    pruned = {k: v for k, v in schema.items() if k not in ("$schema", "additionalProperties")}
    # Ensure top-level structure is an object
    if pruned.get("type") != "object":
        pruned = {"type": "object", "properties": pruned.get("properties", {}), "required": pruned.get("required", [])}
    return pruned


def _extract_text_from_mcp_result(result) -> str:
    """
    Convert an MCP ToolResult into simple text/JSON the model can consume.
    Works with common MCP result shapes (text/json parts).
    """
    # Many MCP servers return result.content = [{ "type": "text", "text": "..."} , ...]
    parts = getattr(result, "content", None)
    if isinstance(parts, list):
        texts = []
        for p in parts:
            # p might be a Pydantic/Typed object
            kind = getattr(p, "type", None) or (isinstance(p, dict) and p.get("type"))
            if kind == "text":
                texts.append(getattr(p, "text", None) or (isinstance(p, dict) and p.get("text")) or "")
            elif kind in ("json", "object", "data"):
                data = getattr(p, "data", None) or (isinstance(p, dict) and p.get("data"))
                texts.append(json.dumps(data, ensure_ascii=False))
        if texts:
            return "\n".join(t for t in texts if t)
    # Fallback: stringify
    try:
        return json.dumps(result, default=lambda o: getattr(o, "__dict__", str(o)), ensure_ascii=False)
    except Exception:
        return str(result)


async def call_llm_with_tools(prompt: str, app_id: str, tenant: str):
    # ---- 2) Connect to the MCP server you control ----
    # Replace command/args with your server process or use SSE transport if that’s how you host it.
    transport = UvxStdioTransport(
        from_package="git+https://github.com/elyxlz/microsoft-mcp.git",
        tool_name="microsoft-mcp",
        env_vars={
            "MICROSOFT_MCP_CLIENT_ID": app_id,
            "MICROSOFT_MCP_TENANT_ID": tenant,
        },
    )
    mcp_client = Client(transport)

    async with mcp_client:

        # Discover server tools, then whitelist
        tools_resp = await mcp_client.list_tools()
        selected = [t for t in tools_resp if t.name in ALLOWED]

        # ---- 3) Convert the selected MCP tools into Gemini function declarations ----
        function_decls = []
        for t in selected:
            # MCP exposes JSON Schema in t.inputSchema; convert to the subset Gemini expects.
            params_schema = _sanitize_openapi_schema(getattr(t, "inputSchema", {}) or {})
            function_decls.append({
                "name": t.name,
                "description": getattr(t, "description", "") or f"MCP tool '{t.name}'.",
                "parameters": params_schema,
            })

        # Build the Gemini config with only your subset. (Extra safety: constrain allowed names.)
        tool = types.Tool(function_declarations=function_decls)
        tool_config = types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode="ANY",  # to force the model to call tools
                allowed_function_names=[fd["name"] for fd in function_decls],
            )
        )

        client = genai.Client()  # picks up GOOGLE_API_KEY / GEMINI_API_KEY env var
        contents = [prompt]

    # Manual tool-call loop: model -> (function_call) -> MCP -> (function_response) -> model
        resp = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=contents,
            config=types.GenerateContentConfig(
                tools=[tool],
                tool_config=tool_config,
                temperature=0,
                system_instruction=system_instruction
            ),
        )

        # Collect all function calls from this turn (the SDK may return multiple)
        function_calls = []
        for cand in resp.candidates or []:
            for part in cand.content.parts or []:
                fc = getattr(part, "function_call", None)
                if fc:
                    function_calls.append(fc)

            if not function_calls:
                # No more calls; return the final user-facing text.
                print(resp.text)
                return resp.text

        # # Execute each requested call via MCP and push results back for the next turn
        # for fc in function_calls:
        # # Guardrail: ensure the call is in our allowed subset
        #     if not any(fc.name == fd["name"] for fd in function_decls):
        #         # If a non-whitelisted call is requested, tell the model it’s unavailable
        #         contents.append(types.ModelContent(parts=[types.Part.from_function_call(name=fc.name, args=fc.args or {})]))
        #         contents.append(types.UserContent(parts=[types.Part.from_function_response(
        #             name=fc.name,
        #             response={"error": "Function not permitted in this context."},
        #         )]))
        #         continue

        #     # Strip namespace to call MCP tool
        #     raw_tool_name = fc.name.split(".", 1)[1] if "." in fc.name else fc.name
        #     mcp_result = await mcp_client.call_tool(raw_tool_name, arguments=(fc.args or {}))

        #     # Add the model’s functionCall *and* our functionResponse as the next turns
        #     contents.append(types.ModelContent(parts=[
        #         types.Part.from_function_call(name=fc.name, args=fc.args or {})
        #     ]))
        #     contents.append(types.UserContent(parts=[
        #         types.Part.from_function_response(
        #             name=fc.name,
        #             response={"result": _extract_text_from_mcp_result(mcp_result)},
        #         )
        #     ]))

    return function_calls, contents
prompt = "Block out my calendar for the entirity of tomorrow. I wanna work on building agents."

llm_tool_calls, contents = await call_llm_with_tools(prompt, APP_ID, TENANT)


In [8]:
llm_tool_calls[0].args


{'account_id': '00000000-0000-0000-b358-02ef6301080d.9188040d-6c67-4c5b-b112-36a304b66dad',
 'end': '2025-09-23T23:59:59',
 'start': '2025-09-23T00:00:00',
 'subject': 'Work on building agents'}

In [ ]:
transport = UvxStdioTransport(
        from_package="git+https://github.com/elyxlz/microsoft-mcp.git",
        tool_name="microsoft-mcp",
        env_vars={
            "MICROSOFT_MCP_CLIENT_ID": APP_ID,
            "MICROSOFT_MCP_TENANT_ID": TENANT,
        },
    )
mcp_client = Client(transport)

async with mcp_client:
    create_event = await mcp_client.call_tool("create_event", llm_tool_calls[0].args)



In [ ]:
def execute_llm_tool_calls(function_calls, function_decls, mcp_session):

    transport = UvxStdioTransport(
        from_package="git+https://github.com/elyxlz/microsoft-mcp.git",
        tool_name="microsoft-mcp",
        env_vars={
            "MICROSOFT_MCP_CLIENT_ID": app_id,
            "MICROSOFT_MCP_TENANT_ID": tenant,
        },
    )
    client = Client(transport)

    async with client:



In [ ]:
async def call_mcp_tool(tool_name: str, tool_input: dict):
    transport = UvxStdioTransport(
        from_package="git+https://github.com/elyxlz/microsoft-mcp.git",
        tool_name="microsoft-mcp",
        env_vars={
            "MICROSOFT_MCP_CLIENT_ID": APP_ID,
            "MICROSOFT_MCP_TENANT_ID": TENANT,
        },
    )
    client = Client(transport)

    async with client:
        result = await client.call_tool(tool_name, tool_input)
        return result

my_emails = await call_mcp_tool("list_emails", {})

✓ Authentication successful!
Signed in as: mnylnworkshopparticipant@outlook.com
Account ID: 00000000-0000-0000-b358-02ef6301080d.9188040d-6c67-4c5b-b112-36a304b66dad